# تقييم مستقل لنموذج BudgetThinker-1.5B
**Independent, reproducible evaluation — RQ1–RQ4**

يجيب هذا الـNotebook تجريبياً عن أربعة أسئلة:
- **RQ1:** هل يولّد النموذج Control Tokens ذاتياً دون حقن من محرك الاستدلال؟
- **RQ2:** هل يلتزم بالميزانية دون حقن؟
- **RQ3:** كم يتحسن Budget Following Ratio مع الحقن الأصلي (K=8)؟
- **RQ4:** ما أثر الحقن على Budget Utilization Ratio والدقة واكتمال الإجابة؟

**مبادئ التنفيذ**
- لا تدريب ولا تعديل للأوزان. النموذج المنشور مُتحقَّق منه بـSHA-256 (مطابق لنقطة RL `TCMv4_8ratio_v1_step_421`).
- قاعدة الحقن منقولة حرفياً من `to_replace/vllm/worker/model_runner.py` (commit `481da90`) ومُختبرة على 378,543 قراراً عشوائياً.
- كل توكن مولَّد يحمل وسم مصدر: `m` (النموذج)، `i` (حقن)، `s` (لاحقة الإيقاف القسري).
- الحالتان A وB تتشاركان البذرة نفسها لكل عيّنة، فتتطابقان حرفياً حتى أول حقن.
- كل وحدة (سؤال × ميزانية) تُحفظ ذرّياً، ويُستأنف التشغيل دون تكرار ما اكتمل.

**التشغيل الموسّع (9,600 توليد) محميّ بالمتغير `APPROVED_FULL_RUN = False`.**

## 0. الإعدادات

In [ ]:
# ==== Parameters (edit here) ====
USE_DRIVE = True                 # persist to Google Drive (one-time authorization popup)
DRIVE_DIR = "/content/drive/MyDrive/BudgetThinker_eval"
LOCAL_DIR = "/content/BudgetThinker_eval"
DO_SMOKE = True                  # fp16-vs-fp32 check + one greedy A/B pair
DO_RQ1_PROBE = True              # 5 questions x {500, 2000} x greedy x {A, B} = 20 generations
DO_PILOT = True                  # 5 questions x {500, 2000} x 2 samples x {A, B} = 40 generations
DO_THROUGHPUT_PROBE = True       # synthetic decode timing/memory incl. 16 rows at 10k context
FAST = True                      # exact GQA-decode attention + preallocated KV cache (src/bt_fast.py)
APPROVED_FULL_RUN = False        # set True ONLY after the pilot report is approved
FULL_RUNS = ["greedy_ref", "sampled"]
CU_PER_HOUR = 1.76               # T4 rate from Colab > Resources; edit if different
GPU_OK = ("T4", "A100")          # approved GPU types (A100 approved by the user on 2026-10-11 for the full run)

In [ ]:
import subprocess, sys, time, os, json, platform
T_SESSION0 = time.time()
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "NONE"); print("Python", sys.version.split()[0], "|", platform.platform())
assert gpu, "No GPU attached: Runtime > Change runtime type > T4 GPU"
if not any(g in gpu for g in GPU_OK):
    raise SystemExit(f"Expected one of {GPU_OK}, got {gpu!r}. Stop to avoid an unapproved GPU.")

In [ ]:
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = DRIVE_DIR
else:
    BASE = LOCAL_DIR
for d in ["configs", "notebooks", "raw_generations", "metrics/grading", "reports/figures", "logs", "src"]:
    os.makedirs(os.path.join(BASE, d), exist_ok=True)
os.chdir(BASE)
print("BASE =", BASE)

## 1. البيئة: المكتبات، الكود الرسمي، بيئة المصحّح المعزولة

In [ ]:
t0 = time.time()
!pip install -q "transformers==4.51.3" "math-verify[antlr4_13_2]" 2>&1 | tail -2
OFFICIAL = "/content/BudgetThinker_official"
OFFICIAL_COMMIT = "481da9045f761ba34899271db846d659110b7065"
if not os.path.exists(OFFICIAL):
    !git clone -q https://github.com/MobileLLM/BudgetThinker {OFFICIAL}
!git -C {OFFICIAL} checkout -q {OFFICIAL_COMMIT} && git -C {OFFICIAL} log -1 --format="official repo @ %H (%ad)"
# Official grader pins (sympy==1.12, antlr4 4.11.1) conflict with torch -> isolated --target dir
GRADER_ENV = "/content/grader_env"
if not os.path.exists(os.path.join(GRADER_ENV, "latex2sympy2.py")):
    !pip install -q --target {GRADER_ENV} "sympy==1.12" "antlr4-python3-runtime==4.11.1" "Pebble==5.1.0" "word2number==1.1" "timeout-decorator==0.5.0" regex 2>&1 | grep -v "WARNING" | tail -2
    !pip install -q --no-deps --target {GRADER_ENV} {OFFICIAL}/evaluation/latex2sympy 2>&1 | tail -1
T_INSTALL = time.time() - t0
print(f"install/setup: {T_INSTALL:.0f}s")

## 2. وحدات الكود (مطابقة للملفات المختبرة محلياً)

In [ ]:
%%writefile src/bt_engine.py
"""BudgetThinker independent evaluation: token-level generation engine (HF transformers).

Re-implements the official BudgetThinker inference control at the token level so that
every generated token carries an unambiguous provenance tag:

    m  sampled by the model
    i  control token injected by the engine (it overwrote the token the model sampled)
    s  engine-appended forced-termination suffix

Official references (MobileLLM/BudgetThinker @ 481da9045f761ba34899271db846d659110b7065):
  * Injection rule: to_replace/vllm/worker/model_runner.py, branch remaining="8ratio"
    (patch for vLLM 0.7.3). Ported literally in `official_rule_reference` and incrementally
    in `RowState._injection`.
  * Prompt: evaluation/eval_tools.py::_8ratio_prompt + PROMPT_TEMPLATES["deepseek3"].
  * Stage-1 length cap: evaluation/remaining_eval_multi_process.py, max_tokens = B + B//50.
  * Forced-termination suffix: same file, stage 2: "\n</think>\n\n**Final Answer**\\boxed".
"""
from __future__ import annotations

import dataclasses
import hashlib
import time
from typing import Optional

import torch

# --- Token ids of Xin-Rui/BudgetThinker-1.5b (verified against its tokenizer) -------------
CTRL_IDS = tuple(range(151665, 151672))  # "\n<remaining>1/8</remaining>\n" .. 7/8
CTRL_SET = frozenset(CTRL_IDS)
CTRL_LABEL = {151665 + i: f"{i + 1}/8" for i in range(7)}
FIRST_CTRL, LAST_CTRL = 151671, 151665  # 7/8 is injected first, 1/8 last
THINK_OPEN, THINK_CLOSE = 151648, 151649
EOS = 151643  # <｜end▁of▁sentence｜>
BOS = 151646  # <｜begin▁of▁sentence｜>
PAD = 151643
FORCED_SUFFIX_TEXT = "\n</think>\n\n**Final Answer**\\boxed"
FORCED_SUFFIX_IDS = (198, 151649, 271, 334, 19357, 21806, 334, 59, 79075)
PROMPT_PREFIX = "Return your final response within \\boxed{}. "


def official_prompt(problem: str, budget: int) -> str:
    """Prompt string exactly as built by the official evaluation for tip=8ratio, deepseek3."""
    ctrl = " , ".join(f"\n<remaining>{k}/8</remaining>\n" for k in range(7, 0, -1))
    tip = (f"\n(Complete thinking within {budget} tokens or fewer, 7 special tokens ( {ctrl} ) "
           "will split the thinking process into 8 parts.)")
    return "<｜User｜>" + PROMPT_PREFIX + problem + tip + "<｜Assistant｜>"


def stage1_cap(budget: int) -> int:
    """Official stage-1 max_tokens: B + B//50."""
    return budget + budget // 50


def injection_delta(budget: int, k: int = 8) -> int:
    return budget // k + 1


def expected_schedule(budget: int, k: int = 8) -> list[tuple[int, int]]:
    """(position, token) pairs the official rule produces when the model emits no control token."""
    d = injection_delta(budget, k)
    out, pos = [], d
    for tok in range(FIRST_CTRL, LAST_CTRL - 1, -1):
        out.append((pos, tok))
        pos += d + 1
    return out


def official_rule_reference(output_token_ids, budget: int, n_parts: int = 8) -> Optional[int]:
    """Literal port of the vLLM patch (remaining='8ratio'). Returns the id that overwrites the
    token sampled at index len(output_token_ids), or None. Used to test the fast version."""
    delta = budget // n_parts + 1
    special_tokens = [151665 + i for i in range(n_parts - 1)]
    last_special_token_idx, last_special_token = None, None
    for idx in range(len(output_token_ids) - 1, -1, -1):
        token_id = output_token_ids[idx]
        if token_id in special_tokens:
            last_special_token_idx = idx
            last_special_token = token_id
            break
    if last_special_token == 151665:
        return None
    if last_special_token_idx is not None:
        distance_to_last_special_token = len(output_token_ids) - last_special_token_idx - 1
        if distance_to_last_special_token == delta:
            return last_special_token - 1
        return None
    last_special_token = 151671 + 1
    if len(output_token_ids) == delta:
        return last_special_token - 1
    return None


def derive_seed(*parts) -> int:
    """Stable 63-bit seed from arbitrary parts (independent of Python hash randomization)."""
    h = hashlib.sha256("|".join(map(str, parts)).encode()).digest()
    return int.from_bytes(h[:8], "big") & ((1 << 63) - 1)


@dataclasses.dataclass
class GenConfig:
    temperature: float = 0.6
    top_p: float = 0.95
    n_parts: int = 8
    natural_answer_cap: int = 512   # tokens allowed after a natural </think>
    forced_answer_cap: int = 50     # tokens allowed after the forced suffix (paper, Sec. 3.1)
    ctrl_trace_threshold: float = 1e-3
    inject_in_answer: bool = True   # official rule keeps running after </think>


@dataclasses.dataclass
class RowSpec:
    gen_id: str
    prompt_ids: list
    budget: int
    inject: bool  # True = Condition B (official injection), False = Condition A
    seed: int
    meta: dict = dataclasses.field(default_factory=dict)


class RowState:
    """Per-generation state machine: think -> (answer | suffix -> forced_answer) -> done."""

    def __init__(self, spec: RowSpec, cfg: GenConfig):
        self.spec, self.cfg = spec, cfg
        self.cap = stage1_cap(spec.budget)
        self.delta = injection_delta(spec.budget, cfg.n_parts)
        self.expected = dict(expected_schedule(spec.budget, cfg.n_parts))
        self.gen = torch.Generator(device="cpu")
        self.gen.manual_seed(spec.seed)
        self.out: list[int] = []
        self.prov: list[str] = []
        self.phase = "think"
        self.done = False
        self.last_ctrl_idx: Optional[int] = None
        self.last_ctrl_tok: Optional[int] = None
        self.ctrl_events: list[dict] = []
        self.ctrl_trace: list[list] = []      # [pos, total ctrl mass, argmax ctrl id, its prob]
        self.expected_probs: list[dict] = []  # model prob of the scheduled token at scheduled pos
        self.overwritten: list[int] = []
        self.think_end: Optional[int] = None
        self.answer_start: Optional[int] = None
        self.termination: Optional[str] = None
        self.stop_reason: Optional[str] = None
        self.suffix_i = 0
        self.t_finish: Optional[float] = None

    # Official rule, O(1) per step; equivalent to official_rule_reference (see tests).
    def _injection(self) -> Optional[int]:
        n = len(self.out)
        if self.last_ctrl_tok == LAST_CTRL:
            return None
        if self.last_ctrl_idx is not None:
            return self.last_ctrl_tok - 1 if n - self.last_ctrl_idx - 1 == self.delta else None
        return FIRST_CTRL if n == self.delta else None

    def _append(self, tok: int, prov: str):
        if tok in CTRL_SET:
            self.last_ctrl_idx, self.last_ctrl_tok = len(self.out), tok
        self.out.append(tok)
        self.prov.append(prov)

    def _finish(self, reason: str):
        self.done, self.stop_reason, self.phase = True, reason, "done"
        self.t_finish = time.time()

    def _log_probs(self, n: int, ctrl_p: list):
        mass = sum(ctrl_p)
        if mass > self.cfg.ctrl_trace_threshold:
            j = max(range(7), key=ctrl_p.__getitem__)
            self.ctrl_trace.append([n, round(mass, 6), CTRL_IDS[j], round(ctrl_p[j], 6)])
        if n in self.expected:
            tok = self.expected[n]
            self.expected_probs.append({"pos": n, "token": tok, "p": ctrl_p[CTRL_IDS.index(tok)],
                                        "phase": self.phase})

    def _sampled_or_injected(self, sampled: int, ctrl_p: list, allow_inject: bool) -> int:
        n = len(self.out)
        self._log_probs(n, ctrl_p)
        tok, prov = sampled, "m"
        forced = self._injection() if (self.spec.inject and allow_inject) else None
        if forced is not None:
            tok, prov = forced, "i"
            self.overwritten.append(sampled)
        if tok in CTRL_SET:
            self.ctrl_events.append({
                "pos": n, "token": tok, "label": CTRL_LABEL[tok],
                "source": "injected" if prov == "i" else "model", "phase": self.phase,
                "p_model": ctrl_p[CTRL_IDS.index(tok)],
                "overwrote": sampled if prov == "i" else None,
            })
        self._append(tok, prov)
        return tok

    def step(self, sampled: int, ctrl_p: list) -> int:
        """Consume the token the model sampled at this step; return the token actually emitted."""
        if self.phase == "think":
            tok = self._sampled_or_injected(sampled, ctrl_p, True)
            if tok == THINK_CLOSE:
                self.think_end = len(self.out) - 1
                self.answer_start = len(self.out)
                self.termination, self.phase = "natural", "answer"
            elif tok == EOS:
                self.termination = "eos_in_think"
                self._finish("eos_in_think")
            elif len(self.out) >= self.cap:
                self.termination, self.phase = "forced", "suffix"
            return tok
        if self.phase == "suffix":
            tok = FORCED_SUFFIX_IDS[self.suffix_i]
            self._append(tok, "s")
            self.suffix_i += 1
            if self.suffix_i == len(FORCED_SUFFIX_IDS):
                self.phase, self.answer_start = "forced_answer", len(self.out)
            return tok
        if self.phase in ("answer", "forced_answer"):
            # Official stage 2 is a fresh call, so the rule cannot fire within 50 forced tokens;
            # after a natural </think> the official rule keeps counting positions.
            allow = self.phase == "answer" and self.cfg.inject_in_answer
            tok = self._sampled_or_injected(sampled, ctrl_p, allow)
            cap = self.cfg.natural_answer_cap if self.phase == "answer" else self.cfg.forced_answer_cap
            if tok == EOS:
                self._finish("natural_eos" if self.phase == "answer" else "forced_eos")
            elif len(self.out) - self.answer_start >= cap:
                self._finish("natural_answer_cap" if self.phase == "answer" else "forced_answer_cap")
            return tok
        raise RuntimeError(f"step() on finished row {self.spec.gen_id}")

    def needs_sample(self) -> bool:
        return self.phase in ("think", "answer", "forced_answer")

    def record(self, tokenizer=None) -> dict:
        s, out = self.spec, self.out
        b = s.budget
        if self.termination == "natural":
            reasoning = self.think_end
        elif self.termination == "forced":
            reasoning = self.cap
        else:  # eos_in_think
            reasoning = len(out) - 1
        think_slice = range(0, reasoning)
        ctrl_inj_think = sum(1 for i in think_slice if self.prov[i] == "i")
        ctrl_self_think = sum(1 for i in think_slice if self.prov[i] == "m" and out[i] in CTRL_SET)
        ctrl_inj_all = self.prov.count("i")
        ctrl_self_all = sum(1 for t, p in zip(out, self.prov) if p == "m" and t in CTRL_SET)
        answer_tokens = len(out) - self.answer_start if self.answer_start is not None else 0
        counts = {
            "generated_total": len(out),
            "reasoning_tokens": reasoning,  # tokens before </think> (incl. <think> and control tokens)
            "reasoning_tokens_excl_ctrl": reasoning - ctrl_inj_think - ctrl_self_think,
            "reasoning_tokens_excl_injected": reasoning - ctrl_inj_think,
            # The model emits BOS before <think> (as in its SFT targets), so look past it.
            "think_open_generated": THINK_OPEN in out[:3],
            "bos_generated": bool(out and out[0] == BOS),
            "ctrl_injected_think": ctrl_inj_think,
            "ctrl_self_think": ctrl_self_think,
            "ctrl_injected_total": ctrl_inj_all,
            "ctrl_self_total": ctrl_self_all,
            "suffix_tokens": self.prov.count("s"),
            "answer_tokens": answer_tokens,
            "stage1_cap": self.cap,
            "suppressed_think_close": sum(1 for t in self.overwritten if t == THINK_CLOSE),
            "suppressed_eos": sum(1 for t in self.overwritten if t == EOS),
        }
        if self.termination == "natural":
            category = "natural_within_budget" if reasoning <= b else "natural_over_budget"
        else:
            category = self.termination
        rec = {
            "gen_id": s.gen_id, "budget": b, "condition": "B" if s.inject else "A", "seed": s.seed,
            **s.meta,
            "prompt_ids": list(s.prompt_ids), "prompt_len": len(s.prompt_ids),
            "output_ids": list(out), "provenance": "".join(self.prov),
            "termination": self.termination, "category": category, "stop_reason": self.stop_reason,
            "think_end_pos": self.think_end, "answer_start": self.answer_start,
            "counts": counts, "ctrl_events": self.ctrl_events, "ctrl_trace": self.ctrl_trace,
            "expected_ctrl_probs": self.expected_probs,
        }
        if tokenizer is not None:
            rec["text"] = tokenizer.decode(out, skip_special_tokens=False)
            rec["answer_text"] = (tokenizer.decode(out[self.answer_start:], skip_special_tokens=False)
                                  if self.answer_start is not None else "")
        return rec


def sample_tokens(logits: torch.Tensor, temperature: float, top_p: float, gens: list):
    """vLLM-style temperature + top-p sampling with one CPU generator per row (inverse CDF).

    Returns (token ids [b] long, diagnostic probs [b, V] float32). For greedy decoding the
    diagnostic probs are softmax(logits) (temperature 1).
    """
    if temperature == 0:
        return logits.argmax(dim=-1), torch.softmax(logits, dim=-1)
    probs = torch.softmax(logits / temperature, dim=-1)
    sp, si = probs.sort(dim=-1, descending=False)
    if top_p < 1.0:
        mask = sp.cumsum(dim=-1) <= (1.0 - top_p)
        mask[:, -1] = False
        sp = sp.masked_fill(mask, 0.0)
    cum = sp.cumsum(dim=-1)
    u = torch.tensor([[torch.rand(1, generator=g).item()] for g in gens],
                     dtype=cum.dtype, device=cum.device) * cum[:, -1:]
    k = torch.searchsorted(cum, u, right=True).clamp_(max=cum.shape[-1] - 1)
    return si.gather(1, k).squeeze(1), probs


@torch.inference_mode()
def run_rows(model, specs: list[RowSpec], cfg: GenConfig, tokenizer=None, max_steps: int = None,
             prealloc: bool = False):
    """Generate all rows in one left-padded batch; finished rows are pruned from the KV cache.

    prealloc=True uses bt_fast.PreallocCache (same values, no per-step torch.cat)."""
    from transformers import DynamicCache

    dev = next(model.parameters()).device
    n = len(specs)
    width = max(len(s.prompt_ids) for s in specs)
    ids = torch.full((n, width), PAD, dtype=torch.long)
    attn = torch.zeros((n, width), dtype=torch.long)
    for r, s in enumerate(specs):
        ids[r, width - len(s.prompt_ids):] = torch.tensor(s.prompt_ids)
        attn[r, width - len(s.prompt_ids):] = 1
    pos = (attn.cumsum(-1) - 1).clamp(min=0)
    ids, attn, pos = ids.to(dev), attn.to(dev), pos.to(dev)
    ctrl_index = torch.tensor(CTRL_IDS, device=dev)

    t0 = time.time()
    if prealloc:
        from bt_fast import PreallocCache
        longest = max(stage1_cap(s.budget) for s in specs) + max(
            cfg.natural_answer_cap, len(FORCED_SUFFIX_IDS) + cfg.forced_answer_cap)
        cache = PreallocCache(width + longest + 2)
    else:
        cache = DynamicCache()
    out = model(input_ids=ids, attention_mask=attn, position_ids=pos, past_key_values=cache,
                use_cache=True, logits_to_keep=1)
    logits = out.logits[:, -1, :].float()
    next_pos = pos[:, -1:] + 1
    t_prefill = time.time() - t0

    states = [RowState(s, cfg) for s in specs]
    active = list(range(n))
    steps = 0
    while active:
        toks, probs = sample_tokens(logits, cfg.temperature, cfg.top_p, [states[j].gen for j in active])
        ctrl_p = probs.index_select(1, ctrl_index).tolist()
        toks = toks.tolist()
        emitted = [states[j].step(toks[k], ctrl_p[k]) for k, j in enumerate(active)]
        steps += 1
        keep = [k for k, j in enumerate(active) if not states[j].done]
        if not keep or (max_steps is not None and steps >= max_steps):
            break
        if len(keep) < len(active):
            sel = torch.tensor(keep, device=dev)
            cache.batch_select_indices(sel)
            attn, next_pos = attn.index_select(0, sel), next_pos.index_select(0, sel)
            active = [active[k] for k in keep]
            emitted = [emitted[k] for k in keep]
        feed = torch.tensor(emitted, device=dev).unsqueeze(1)
        attn = torch.cat([attn, attn.new_ones((attn.shape[0], 1))], dim=1)
        out = model(input_ids=feed, attention_mask=attn, position_ids=next_pos, past_key_values=cache,
                    use_cache=True, logits_to_keep=1)
        logits = out.logits[:, -1, :].float()
        next_pos = next_pos + 1
    wall = time.time() - t0

    recs = []
    for st in states:
        rec = st.record(tokenizer)
        rec["timing"] = {"batch_wall_s": round(wall, 3), "prefill_s": round(t_prefill, 3),
                         "batch_steps": steps, "batch_rows": n,
                         "finish_offset_s": round(st.t_finish - t0, 3) if st.t_finish else None}
        recs.append(rec)
    return recs

In [ ]:
%%writefile src/bt_fast.py
"""Exact speed-ups for long-context decoding with transformers 4.51.3 on a T4.

1. GQA decode attention. transformers 4.51.3's `sdpa_attention_forward` calls `repeat_kv` on every
   step, materialising the KV cache 6x (12 query heads over 2 KV heads) per layer. For one-token
   decode steps we instead view the 12 query heads as 2 groups x 6 and attend each group to its KV
   head directly. Prefill (q_len > 1) still uses the original SDPA function. The mask semantics stay
   those of `_attn_implementation == "sdpa"` because we replace the function under that key.
2. Preallocated KV cache. `DynamicCache.update` concatenates the whole cache every step; this cache
   writes into a buffer allocated once per batch and returns views.

Both change only how the same quantities are computed (verified against the stock path in
tests/test_fast.py); decoding semantics, sampling and the injection rule are untouched.
"""
from __future__ import annotations

import torch
from transformers import DynamicCache
from transformers.integrations.sdpa_attention import sdpa_attention_forward as _stock_sdpa
from transformers.modeling_utils import ALL_ATTENTION_FUNCTIONS


def gqa_sdpa_forward(module, query, key, value, attention_mask, dropout=0.0, scaling=None,
                     is_causal=None, **kwargs):
    groups = getattr(module, "num_key_value_groups", 1)
    if query.shape[2] != 1 or groups == 1 or dropout:
        return _stock_sdpa(module, query, key, value, attention_mask, dropout=dropout, scaling=scaling,
                           is_causal=is_causal, **kwargs)
    b, h, _, d = query.shape
    kvh, length = key.shape[1], key.shape[2]
    scale = scaling if scaling is not None else d ** -0.5
    # Scale q before the fp16 matmul (as SDPA does): Qwen2's large q/k activations overflow fp16
    # (max 65504) if q.k is formed first and scaled afterwards.
    q = (query * scale).reshape(b, kvh, groups, d)            # head index = kv * groups + g (as repeat_kv)
    scores = torch.matmul(q, key.transpose(-1, -2)).float()   # [b, kvh, groups, L]
    if attention_mask is not None:
        scores = scores + attention_mask[:, :, :, :length].float()      # [b, 1, 1, L] broadcasts
    probs = torch.softmax(scores, dim=-1).to(value.dtype)
    out = torch.matmul(probs, value)                          # [b, kvh, groups, d]
    return out.reshape(b, h, 1, d).transpose(1, 2).contiguous(), None


def enable_gqa_decode():
    ALL_ATTENTION_FUNCTIONS["sdpa"] = gqa_sdpa_forward


def disable_gqa_decode():
    if "sdpa" in ALL_ATTENTION_FUNCTIONS._local_mapping:
        del ALL_ATTENTION_FUNCTIONS["sdpa"]


class PreallocCache(DynamicCache):
    """DynamicCache that writes into per-layer buffers of `max_len` positions instead of torch.cat."""

    def __init__(self, max_len: int):
        super().__init__()
        self.max_len = max_len
        self._lens: list[int] = []

    def update(self, key_states, value_states, layer_idx, cache_kwargs=None):
        n = key_states.shape[-2]
        if layer_idx == 0:
            self._seen_tokens += n
        if len(self.key_cache) <= layer_idx:
            b, h, _, d = key_states.shape
            kb = key_states.new_empty((b, h, self.max_len, d))
            vb = value_states.new_empty((b, h, self.max_len, d))
            kb[:, :, :n] = key_states
            vb[:, :, :n] = value_states
            self.key_cache.append(kb)
            self.value_cache.append(vb)
            self._lens.append(n)
        else:
            s = self._lens[layer_idx]
            if s + n > self.max_len:
                raise RuntimeError(f"PreallocCache overflow: {s + n} > {self.max_len}")
            self.key_cache[layer_idx][:, :, s:s + n] = key_states
            self.value_cache[layer_idx][:, :, s:s + n] = value_states
            self._lens[layer_idx] = s + n
        cur = self._lens[layer_idx]
        return self.key_cache[layer_idx][:, :, :cur], self.value_cache[layer_idx][:, :, :cur]

    def get_seq_length(self, layer_idx: int = 0) -> int:
        return self._lens[layer_idx] if layer_idx < len(self._lens) else 0

    def batch_select_indices(self, indices: torch.Tensor):
        for i in range(len(self.key_cache)):
            self.key_cache[i] = self.key_cache[i].index_select(0, indices)
            self.value_cache[i] = self.value_cache[i].index_select(0, indices)

In [ ]:
%%writefile src/bt_runner.py
"""BudgetThinker independent evaluation: question selection, job units, resumable execution.

A *unit* is one (question, budget) pair with all its rows: n samples x conditions (A, B).
Conditions A and B of the same sample index share one seed (common random numbers), so the two
generations are token-identical until the first injected control token.
Each finished unit is written atomically to raw_generations/<run>/B<budget>/<unit_id>.jsonl.gz;
units with an existing file are skipped on restart.
"""
from __future__ import annotations

import gzip
import json
import logging
import os
import platform
import random
import time

import torch

import bt_engine as E


def load_math500(path: str) -> list[dict]:
    return [json.loads(l) for l in open(path, encoding="utf-8")]


def select_questions(rows: list[dict], n: int, seed: int) -> list[dict]:
    """Stratified by MATH level (equal share per level), deterministic given seed."""
    rng = random.Random(seed)
    by_level: dict[int, list[dict]] = {}
    for r in rows:
        by_level.setdefault(int(r["level"]), []).append(r)
    levels = sorted(by_level)
    per = [n // len(levels) + (1 if i < n % len(levels) else 0) for i in range(len(levels))]
    chosen = []
    for lvl, k in zip(levels, per):
        pool = sorted(by_level[lvl], key=lambda r: r["unique_id"])
        chosen += rng.sample(pool, k)
    order = rng.sample(range(len(chosen)), len(chosen))  # interleave levels for early-stop balance
    return [chosen[i] for i in order]


def make_units(questions, budgets, conditions, n_samples, run, temperature, top_p, tokenizer, meta):
    units = []
    for q in questions:
        for b in budgets:
            prompt = E.official_prompt(q["problem"], b)
            ids = tokenizer(prompt).input_ids  # adds BOS, as vLLM does for a string prompt
            uid = f"{q['unique_id'].replace('/', '_').replace('.json', '')}_B{b}"
            rows = []
            for s in range(n_samples):
                seed = E.derive_seed(run, q["unique_id"], b, s)
                for c in conditions:
                    rows.append(E.RowSpec(
                        gen_id=f"{run}|{uid}|s{s}|{c}", prompt_ids=ids, budget=b, inject=(c == "B"),
                        seed=seed,
                        meta={**meta, "run": run, "unit_id": uid, "q_uid": q["unique_id"],
                              "q_level": int(q["level"]), "q_subject": q["subject"],
                              "gt_answer": q["answer"], "sample_idx": s, "prompt_text": prompt,
                              "temperature": temperature, "top_p": top_p if temperature > 0 else 1.0}))
            units.append({"unit_id": uid, "budget": b, "rows": rows, "q": q})
    return units


def unit_path(out_dir: str, run: str, unit: dict) -> str:
    return os.path.join(out_dir, run, f"B{unit['budget']}", unit["unit_id"] + ".jsonl.gz")


def write_unit(path: str, records: list[dict]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + ".tmp"
    with gzip.open(tmp, "wt", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    os.replace(tmp, path)


def read_units(out_dir: str, run: str) -> list[dict]:
    recs = []
    root = os.path.join(out_dir, run)
    for dp, _, fs in os.walk(root):
        for fn in sorted(fs):
            if fn.endswith(".jsonl.gz"):
                with gzip.open(os.path.join(dp, fn), "rt", encoding="utf-8") as f:
                    recs += [json.loads(l) for l in f]
    return recs


def env_info() -> dict:
    import transformers
    info = {"python": platform.python_version(), "torch": torch.__version__,
            "transformers": transformers.__version__, "cuda": torch.version.cuda}
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        info.update(gpu=p.name, gpu_mem_gb=round(p.total_memory / 2**30, 2),
                    capability=f"{p.major}.{p.minor}")
    return info


def run_units(model, tokenizer, units, cfg: E.GenConfig, out_dir: str, run: str,
              rows_per_batch: dict, log: logging.Logger, max_units: int | None = None,
              prealloc: bool = False) -> dict:
    """Run pending units grouped by budget; returns timing stats for this call."""
    pending = [u for u in units if not os.path.exists(unit_path(out_dir, run, u))]
    n_done = len(units) - len(pending)
    if max_units is not None:
        pending = pending[:max_units]
    log.info(f"[{run}] {len(units)} units, {n_done} already done, {len(pending)} to run now")
    stats = {"batches": [], "gen_tokens": 0, "wall_s": 0.0}
    by_budget: dict[int, list] = {}
    for u in pending:
        by_budget.setdefault(u["budget"], []).append(u)
    for b in sorted(by_budget):
        queue = by_budget[b]
        cap = rows_per_batch.get(b, rows_per_batch.get("default", 16))
        while queue:
            group, n_rows = [], 0
            while queue and (not group or n_rows + len(queue[0]["rows"]) <= cap):
                n_rows += len(queue[0]["rows"])
                group.append(queue.pop(0))
            specs = [r for u in group for r in u["rows"]]
            if torch.cuda.is_available():
                torch.cuda.reset_peak_memory_stats()
            t0 = time.time()
            try:
                recs = E.run_rows(model, specs, cfg, tokenizer, prealloc=prealloc)
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                if len(group) == 1 and len(specs) > 1:
                    log.warning(f"OOM at B={b} with {len(specs)} rows; splitting the unit in halves")
                    half = len(specs) // 2
                    recs = E.run_rows(model, specs[:half], cfg, tokenizer, prealloc=prealloc)
                    torch.cuda.empty_cache()
                    recs += E.run_rows(model, specs[half:], cfg, tokenizer, prealloc=prealloc)
                else:
                    log.warning(f"OOM at B={b} with {len(group)} units; requeueing one unit per batch")
                    for u in reversed(group):
                        queue.insert(0, u)
                    rows_per_batch[b] = max(len(group[0]["rows"]), cap // 2)
                    cap = rows_per_batch[b]
                    continue
            wall = time.time() - t0
            peak = torch.cuda.max_memory_allocated() / 2**30 if torch.cuda.is_available() else 0.0
            ntok = sum(len(r["output_ids"]) for r in recs)
            gpu = env_info().get("gpu", "cpu")
            by_id = {r["gen_id"]: r for r in recs}
            for u in group:
                urecs = [by_id[s.gen_id] for s in u["rows"]]
                for r in urecs:
                    r["timing"]["gpu"] = gpu
                    r["timing"]["peak_mem_gb"] = round(peak, 2)
                write_unit(unit_path(out_dir, run, u), urecs)
            row = {"budget": b, "units": len(group), "rows": len(specs), "wall_s": round(wall, 2),
                   "gen_tokens": ntok, "tok_per_s": round(ntok / wall, 1), "peak_mem_gb": round(peak, 2),
                   "max_len": max(len(r["output_ids"]) for r in recs)}
            stats["batches"].append(row)
            stats["gen_tokens"] += ntok
            stats["wall_s"] += wall
            log.info(f"[{run}] B={b} units={len(group)} rows={len(specs)} wall={wall:.1f}s "
                     f"tokens={ntok} ({ntok / wall:.0f} tok/s) peak={peak:.2f}GB")
    return stats


def make_logger(path: str) -> logging.Logger:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    log = logging.getLogger("bt")
    log.setLevel(logging.INFO)
    log.propagate = False  # Colab's root handler would print every line twice
    if not any(isinstance(h, logging.FileHandler) and h.baseFilename == os.path.abspath(path)
               for h in log.handlers):
        fh = logging.FileHandler(path, encoding="utf-8")
        fh.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
        log.addHandler(fh)
    if not any(type(h) is logging.StreamHandler for h in log.handlers):
        sh = logging.StreamHandler()
        sh.setFormatter(logging.Formatter("%(asctime)s %(message)s", "%H:%M:%S"))
        log.addHandler(sh)
    return log

In [ ]:
%%writefile src/bt_metrics.py
"""BudgetThinker independent evaluation: grading, metrics, statistics, plots.

Uncertainty: questions are the sampling unit. All 95% CIs are cluster (question-level) percentile
bootstraps; A-vs-B comparisons are paired at the question level (same questions, same seeds) with a
sign-flip permutation test on per-question differences and Holm correction across budgets.
"""
from __future__ import annotations

import json
import os
import subprocess
import sys

import numpy as np
import pandas as pd

import bt_engine as E

# ----------------------------------------------------------------------------- grading
def grading_text(rec: dict, tokenizer) -> str:
    ids = rec["output_ids"]
    if ids and ids[-1] == E.EOS:
        ids = ids[:-1]
    return tokenizer.decode(ids, skip_special_tokens=False)


def grade_official(records, tokenizer, solutions: dict, eval_dir: str, grader_env: str, work_dir: str):
    """Run the official parser/grader in its isolated environment; returns {gen_id: result}."""
    os.makedirs(work_dir, exist_ok=True)
    inp, out = os.path.join(work_dir, "grade_in.jsonl"), os.path.join(work_dir, "grade_out.jsonl")
    with open(inp, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps({"gen_id": r["gen_id"], "text": grading_text(r, tokenizer),
                                "solution": solutions[r["q_uid"]]}, ensure_ascii=False) + "\n")
    env = {**os.environ, "PYTHONPATH": grader_env}
    script = os.path.join(os.path.dirname(os.path.abspath(__file__)), "bt_grade.py")
    p = subprocess.run([sys.executable, script, eval_dir, inp, out], env=env, cwd=work_dir,
                       capture_output=True, text=True)
    if p.returncode != 0:
        raise RuntimeError(p.stderr[-3000:])
    print(p.stdout.strip().splitlines()[-1])
    return {d["gen_id"]: d for d in map(json.loads, open(out, encoding="utf-8"))}


def answer_segment(rec: dict) -> str:
    """Text after </think>. For forced stops the engine suffix ends in '\\boxed', which is not part
    of answer_text (that starts after the suffix), so it is restored here."""
    seg = rec.get("answer_text") or ""
    return "**Final Answer**\\boxed" + seg if rec.get("termination") == "forced" else seg


def grade_math_verify(records, tokenizer, gold: dict) -> dict:
    """Secondary grader (HF Math-Verify) on the answer segment; None if unavailable."""
    try:
        from math_verify import parse, verify
    except ImportError:
        return {}
    res = {}
    for r in records:
        seg = answer_segment(r)
        try:
            g = parse("$" + gold[r["q_uid"]] + "$")
            res[r["gen_id"]] = bool(verify(g, parse(seg))) if seg else False
        except Exception:
            res[r["gen_id"]] = None
    return res


def has_balanced_boxed(s: str) -> bool:
    i = s.rfind("boxed")
    if i < 0 or i + 5 >= len(s) or s[i + 5] != "{":
        return False
    depth = 0
    for ch in s[i + 5:]:
        depth += (ch == "{") - (ch == "}")
        if depth == 0:
            return True
    return False


# ----------------------------------------------------------------------------- flatten
def flatten(records, official: dict, mv: dict) -> pd.DataFrame:
    rows = []
    for r in records:
        c, b = r["counts"], r["budget"]
        g = official.get(r["gen_id"], {})
        ans = answer_segment(r)
        rows.append({
            "gen_id": r["gen_id"], "run": r["run"], "q_uid": r["q_uid"], "q_level": r["q_level"],
            "budget": b, "condition": r["condition"], "sample_idx": r["sample_idx"], "seed": r["seed"],
            "category": r["category"], "termination": r["termination"], "stop_reason": r["stop_reason"],
            "natural_within": r["category"] == "natural_within_budget",
            "natural_any": r["termination"] == "natural",
            "forced": r["termination"] == "forced",
            "eos_in_think": r["termination"] == "eos_in_think",
            "reasoning_tokens": c["reasoning_tokens"], "reasoning_ratio": c["reasoning_tokens"] / b,
            "reasoning_excl_ctrl": c["reasoning_tokens_excl_ctrl"],
            "answer_tokens": c["answer_tokens"], "generated_total": c["generated_total"],
            "total_ratio": (c["reasoning_tokens"] + c["answer_tokens"]) / b,
            "ctrl_injected": c["ctrl_injected_total"], "ctrl_self": c["ctrl_self_total"],
            "ctrl_self_think": c["ctrl_self_think"], "ctrl_injected_think": c["ctrl_injected_think"],
            "suppressed_think_close": c["suppressed_think_close"], "suppressed_eos": c["suppressed_eos"],
            "think_open_generated": E.THINK_OPEN in r["output_ids"][:3],  # BOS may come first
            "bos_generated": bool(r["output_ids"]) and r["output_ids"][0] == E.BOS,
            "pred": g.get("pred"), "gt": g.get("gt"), "correct": bool(g.get("correct", False)),
            "grader_timeout": g.get("timeout"), "pred_from_boxed": g.get("pred_from_boxed"),
            "correct_mv": mv.get(r["gen_id"]),
            "answer_complete": has_balanced_boxed(ans) and r["stop_reason"] in ("natural_eos", "forced_eos"),
            "ctrl_in_answer": sum(1 for e in r["ctrl_events"] if e["phase"] == "answer"),
            "wall_s": r["timing"]["batch_wall_s"], "gpu": r["timing"].get("gpu"),
        })
    df = pd.DataFrame(rows)
    df["grader_disagree"] = df["correct_mv"].notna() & (df["correct_mv"] != df["correct"])
    return df


# ----------------------------------------------------------------------------- statistics
INDICATORS = {"BFR": "natural_within", "BFR_slack": "natural_any", "FTR": "forced",
              "EOS_in_think": "eos_in_think", "Accuracy": "correct", "Answer_complete": "answer_complete"}


def _per_question(d: pd.DataFrame, value: str, where: str | None = None):
    x = d if where is None else d[d[where]]
    g = x.groupby("q_uid")[value].agg(["sum", "count"])
    return g.reindex(sorted(d["q_uid"].unique())).fillna(0.0)


def cluster_boot(num: np.ndarray, den: np.ndarray, reps=2000, seed=0):
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(num), size=(reps, len(num)))
    with np.errstate(invalid="ignore", divide="ignore"):
        stats = num[idx].sum(1) / den[idx].sum(1)
    est = num.sum() / den.sum() if den.sum() > 0 else np.nan
    lo, hi = np.nanpercentile(stats, [2.5, 97.5]) if np.isfinite(stats).any() else (np.nan, np.nan)
    return est, lo, hi


def icc_binary(d: pd.DataFrame, col: str) -> float:
    """One-way ANOVA ICC for a 0/1 outcome clustered by question."""
    g = d.groupby("q_uid")[col]
    k = g.size().mean()
    means, grand = g.mean(), d[col].mean()
    n_q = g.ngroups
    if n_q < 2 or k < 2:
        return np.nan
    msb = k * ((means - grand) ** 2).sum() / (n_q - 1)
    msw = g.var(ddof=1).sum() / n_q
    return float((msb - msw) / (msb + (k - 1) * msw)) if (msb + (k - 1) * msw) > 0 else np.nan


def summarize(df: pd.DataFrame, reps=2000) -> pd.DataFrame:
    out = []
    for (run, b, c), d in df.groupby(["run", "budget", "condition"]):
        row = {"run": run, "budget": b, "condition": c, "n_gen": len(d), "n_q": d["q_uid"].nunique()}
        for name, col in INDICATORS.items():
            pq = _per_question(d.assign(_v=d[col].astype(float)), "_v")
            row[name], row[name + "_lo"], row[name + "_hi"] = cluster_boot(pq["sum"].values, pq["count"].values, reps)
        # BUR: reasoning tokens / B among natural-within-budget generations (paper Fig. 3b)
        for name, col in (("BUR", "reasoning_ratio"), ("BUR_total", "total_ratio")):
            pq = _per_question(d, col, "natural_within")
            row[name], row[name + "_lo"], row[name + "_hi"] = cluster_boot(pq["sum"].values, pq["count"].values, reps)
        pq = _per_question(d, "reasoning_tokens")
        row["MeanReasoning"], row["MeanReasoning_lo"], row["MeanReasoning_hi"] = cluster_boot(
            pq["sum"].values, pq["count"].values, reps)
        row["MedianReasoning"] = float(d["reasoning_tokens"].median())
        row["Acc_natural"] = d.loc[d["natural_any"], "correct"].mean()
        row["Acc_forced"] = d.loc[d["forced"], "correct"].mean()
        row["mean_ctrl_self"] = d["ctrl_self"].mean()
        row["frac_gen_with_self_ctrl"] = (d["ctrl_self"] > 0).mean()
        row["mean_ctrl_injected"] = d["ctrl_injected"].mean()
        row["suppressed_think_close_rate"] = (d["suppressed_think_close"] > 0).mean()
        row["ctrl_in_answer_rate"] = (d["ctrl_in_answer"] > 0).mean()
        row["pred_not_boxed"] = (d["pred_from_boxed"] == False).mean()  # noqa: E712
        row["grader_disagree"] = d["grader_disagree"].mean()
        row["ICC_correct"] = icc_binary(d.assign(_c=d["correct"].astype(float)), "_c")
        row["ICC_BFR"] = icc_binary(d.assign(_c=d["natural_within"].astype(float)), "_c")
        out.append(row)
    return pd.DataFrame(out)


def holm(p: list[float]) -> list[float]:
    p = np.asarray(p, dtype=float)
    order = np.argsort(p)
    adj = np.empty_like(p)
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (len(p) - rank) * p[i])
        adj[i] = min(1.0, running)
    return adj.tolist()


def sign_flip_p(dq: np.ndarray, reps=20000, seed=0) -> float:
    dq = dq[np.isfinite(dq)]
    if len(dq) == 0 or np.allclose(dq, 0):
        return 1.0
    rng = np.random.default_rng(seed)
    obs = abs(dq.mean())
    flips = rng.choice([-1.0, 1.0], size=(reps, len(dq)))
    return float((np.abs((flips * dq).mean(1)) >= obs - 1e-12).mean())


def paired_ab(df: pd.DataFrame, reps=2000) -> pd.DataFrame:
    """B - A per budget, paired by question; cluster bootstrap CI + sign-flip p (Holm over budgets)."""
    metrics = {"BFR": ("natural_within", None), "FTR": ("forced", None), "Accuracy": ("correct", None),
               "Answer_complete": ("answer_complete", None), "BUR": ("reasoning_ratio", "natural_within"),
               "MeanReasoning": ("reasoning_tokens", None)}
    out = []
    for run, dr in df.groupby("run"):
        for name, (col, where) in metrics.items():
            rows = []
            for b, d in dr.groupby("budget"):
                qs = sorted(d["q_uid"].unique())
                num, den = {}, {}
                for c in ("A", "B"):
                    x = d[d["condition"] == c].assign(_v=lambda t: t[col].astype(float))
                    pq = _per_question(x, "_v", where).reindex(qs).fillna(0.0)
                    num[c], den[c] = pq["sum"].values, pq["count"].values
                rng = np.random.default_rng(1)
                idx = rng.integers(0, len(qs), size=(reps, len(qs)))
                with np.errstate(invalid="ignore", divide="ignore"):
                    boot = (num["B"][idx].sum(1) / den["B"][idx].sum(1)) - (num["A"][idx].sum(1) / den["A"][idx].sum(1))
                    est = num["B"].sum() / den["B"].sum() - num["A"].sum() / den["A"].sum()
                    dq = num["B"] / den["B"] - num["A"] / den["A"]
                lo, hi = np.nanpercentile(boot, [2.5, 97.5])
                rows.append({"run": run, "metric": name, "budget": b, "A": num["A"].sum() / max(den["A"].sum(), 1e-9),
                             "B": num["B"].sum() / max(den["B"].sum(), 1e-9), "diff_B_minus_A": est,
                             "ci_lo": lo, "ci_hi": hi, "n_q_paired": int(np.isfinite(dq).sum()),
                             "p_signflip": sign_flip_p(dq)})
            adj = holm([r["p_signflip"] for r in rows])
            for r, a in zip(rows, adj):
                r["p_holm"] = a
            out += rows
    return pd.DataFrame(out)


def ctrl_positions(records) -> pd.DataFrame:
    """One row per control token occurrence (both sources) with alignment to the ideal schedule."""
    rows = []
    for r in records:
        b = r["budget"]
        exp = {tok: pos for pos, tok in E.expected_schedule(b)}
        self_seq = [e["token"] for e in r["ctrl_events"] if e["source"] == "model"]
        ordered = all(x > y for x, y in zip(self_seq, self_seq[1:]))
        for e in r["ctrl_events"]:
            rows.append({"run": r["run"], "gen_id": r["gen_id"], "budget": b, "condition": r["condition"],
                         "source": e["source"], "label": e["label"], "token": e["token"], "pos": e["pos"],
                         "rel_pos": e["pos"] / b, "expected_pos": exp[e["token"]],
                         "dev_tokens": e["pos"] - exp[e["token"]], "dev_rel": (e["pos"] - exp[e["token"]]) / b,
                         "phase": e["phase"], "p_model": e["p_model"], "self_seq_ordered": ordered})
    return pd.DataFrame(rows)


def expected_prob_table(records) -> pd.DataFrame:
    rows = [{"run": r["run"], "budget": r["budget"], "condition": r["condition"], "pos": e["pos"],
             "label": E.CTRL_LABEL[e["token"]], "p": e["p"]}
            for r in records for e in r["expected_ctrl_probs"]]
    return pd.DataFrame(rows)


# ----------------------------------------------------------------------------- plots
COL = {"A": "#C2410C", "B": "#1D4ED8"}
NAME = {"A": "A: no external control tokens", "B": "B: official injection (K=8)"}


def _line(ax, s, metric, ylabel, pct=True):
    for c in ("A", "B"):
        d = s[s["condition"] == c].sort_values("budget")
        k = 100 if pct else 1
        ax.errorbar(d["budget"], d[metric] * k, yerr=[(d[metric] - d[metric + "_lo"]) * k, (d[metric + "_hi"] - d[metric]) * k],
                    marker="o", ms=5, lw=1.8, capsize=3, color=COL[c], label=NAME[c])
    ax.set_xscale("log")
    ax.set_xticks(sorted(s["budget"].unique()))
    ax.set_xticklabels([str(x) for x in sorted(s["budget"].unique())])
    ax.set_xlabel("Thinking budget B (tokens)")
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3)
    ax.legend(frameon=False, fontsize=8)


def make_plots(summary: pd.DataFrame, df: pd.DataFrame, ctrl: pd.DataFrame, out_dir: str, run: str):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    os.makedirs(out_dir, exist_ok=True)
    s = summary[summary["run"] == run]
    paths = []
    for metric, ylabel, fn, pct in (("BFR", "Budget Following Ratio (%)", "bfr", True),
                                    ("BUR", "Budget Utilization Ratio (%)", "bur", True),
                                    ("Accuracy", "Accuracy (%)", "accuracy", True),
                                    ("FTR", "Forced Termination Rate (%)", "ftr", True)):
        fig, ax = plt.subplots(figsize=(6, 4))
        _line(ax, s, metric, ylabel, pct)
        ax.set_title(f"{ylabel.split(' (')[0]} vs budget  [{run}]", fontsize=10)
        fig.tight_layout()
        p = os.path.join(out_dir, f"{run}_{fn}.png")
        fig.savefig(p, dpi=150)
        plt.close(fig)
        paths.append(p)
    d = df[df["run"] == run]
    budgets = sorted(d["budget"].unique())
    fig, axes = plt.subplots(1, len(budgets), figsize=(3.2 * len(budgets), 3.4), squeeze=False)
    for ax, b in zip(axes[0], budgets):
        for c in ("A", "B"):
            x = d[(d["budget"] == b) & (d["condition"] == c)]["reasoning_ratio"]
            ax.hist(x, bins=np.linspace(0, 1.05, 22), alpha=0.55, color=COL[c], label=c)
        ax.axvline(1.0, color="k", lw=0.8, ls="--")
        ax.set_title(f"B={b}", fontsize=9)
        ax.set_xlabel("reasoning tokens / B")
    axes[0][0].set_ylabel("generations")
    axes[0][0].legend(frameon=False)
    fig.suptitle(f"Reasoning length distribution [{run}]", fontsize=10)
    fig.tight_layout()
    p = os.path.join(out_dir, f"{run}_length_dist.png")
    fig.savefig(p, dpi=150)
    plt.close(fig)
    paths.append(p)
    cs = ctrl[(ctrl["run"] == run) & (ctrl["source"] == "model")] if len(ctrl) else ctrl
    fig, ax = plt.subplots(figsize=(7, 3.6))
    if len(cs):
        for lab in sorted(cs["label"].unique(), reverse=True):
            x = cs[cs["label"] == lab]["rel_pos"]
            ax.hist(x, bins=np.linspace(0, 1.1, 45), alpha=0.6, label=f"self {lab}")
        for k in range(1, 8):
            ax.axvline(k / 8, color="grey", lw=0.6, ls=":")
        ax.legend(frameon=False, fontsize=7, ncol=4)
    else:
        ax.text(0.5, 0.5, "No self-generated control tokens observed", ha="center", va="center",
                transform=ax.transAxes)
    ax.set_xlabel("position / B  (dotted: ideal k/8 marks)")
    ax.set_ylabel("count")
    ax.set_title(f"Positions of self-generated control tokens [{run}]", fontsize=10)
    fig.tight_layout()
    p = os.path.join(out_dir, f"{run}_self_ctrl_positions.png")
    fig.savefig(p, dpi=150)
    plt.close(fig)
    paths.append(p)
    return paths

In [ ]:
%%writefile src/bt_grade.py
"""Grade generations with the official BudgetThinker evaluation code (Qwen2.5-Math parser/grader).

Runs as a subprocess inside an isolated package dir (sympy==1.12, antlr4-python3-runtime==4.11.1,
latex2sympy2 from the repo), because those pins conflict with the torch environment.

Usage:
    python bt_grade.py <official_eval_dir> <in.jsonl> <out.jsonl>
Input lines:  {"gen_id", "text", "solution"}   (text = generated output, EOS removed)
Output lines: {"gen_id", "pred", "gt", "correct", "timeout", "pred_from_boxed"}

Mirrors evaluation/remaining_eval_multi_process.py + evaluate.py at commit 481da90:
    pred = strip_string(extract_answer(code, "math500"))         # parser.run_execute
    gt   = extract_answer(solution.strip(), "math500")           # evaluate() -> parse_ground_truth
    correct = math_equal(pred, gt) with a 3 s timeout (pebble), timeout -> False
"""
import json
import sys

EVAL_DIR, IN_PATH, OUT_PATH = sys.argv[1:4]
sys.path.insert(0, EVAL_DIR)

from concurrent.futures import TimeoutError  # noqa: E402

from pebble import ProcessPool  # noqa: E402

from grader import math_equal_process  # noqa: E402  (official)
from parser import extract_answer, run_execute  # noqa: E402  (official)


def main():
    rows = [json.loads(l) for l in open(IN_PATH, encoding="utf-8")]
    preds, gts = [], []
    for r in rows:
        pred, _ = run_execute(None, r["text"], "deepseek3", "math500")
        preds.append(pred)
        gts.append(extract_answer(r["solution"].strip(), "math500"))
    params = [(i, p, g) for i, (p, g) in enumerate(zip(preds, gts))]
    scores, timeouts = [], []
    with ProcessPool(max_workers=1) as pool:
        it = pool.map(math_equal_process, params, timeout=3).result()
        while True:
            try:
                scores.append(bool(next(it)))
                timeouts.append(False)
            except StopIteration:
                break
            except TimeoutError:
                scores.append(False)
                timeouts.append(True)
    with open(OUT_PATH, "w", encoding="utf-8") as f:
        for r, p, g, s, t in zip(rows, preds, gts, scores, timeouts):
            f.write(json.dumps({"gen_id": r["gen_id"], "pred": p, "gt": g, "correct": s, "timeout": t,
                                "pred_from_boxed": "boxed" in r["text"]}, ensure_ascii=False) + "\n")
    print(f"graded {len(rows)}: correct {sum(scores)}, timeouts {sum(timeouts)}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile configs/experiment.json
{
  "study": "Independent evaluation of BudgetThinker-1.5B (RQ1-RQ4)",
  "model": {
    "id": "Xin-Rui/BudgetThinker-1.5b",
    "revision": "b7304c41480eca765e2f77b5a6641133c64c1ea7",
    "tokenizer_id": "Xin-Rui/BudgetThinker-1.5b",
    "tokenizer_revision": "b7304c41480eca765e2f77b5a6641133c64c1ea7",
    "safetensors_sha256": "11441b12d464440d1823825826d7c5b3799ee31e7112d301d7ac7c517efcc6c0",
    "identical_to": "Xin-Rui/BudgetThinker-Training-Stage-Model@f07bbaff:TCMv4_8ratio_v1_step_421_reward_0.838-models/model.safetensors (RL checkpoint; the SFT-only checkpoint TCMv4_8ratio_FULL_1_5B_6epoch has a different hash)",
    "published_dtype": "bfloat16",
    "run_dtype": "bfloat16 on bf16-capable GPUs (A100, compute capability >= 8.0); float16 on T4",
    "run_dtype_reason": "bfloat16 is the published dtype; NVIDIA T4 (sm_75) has no native bfloat16, so the 10-question T4 run used float16"
  },
  "official_code": {
    "repo": "https://github.com/MobileLLM/BudgetThinker",
    "commit": "481da9045f761ba34899271db846d659110b7065",
    "injection_rule": "to_replace/vllm/worker/model_runner.py, remaining='8ratio' (vLLM 0.7.3 patch)",
    "prompt": "evaluation/eval_tools.py::_8ratio_prompt with PROMPT_TEMPLATES['deepseek3']",
    "grader": "evaluation/parser.py::run_execute + evaluation/grader.py::math_equal (Qwen2.5-Math)"
  },
  "dataset": {
    "id": "HuggingFaceH4/MATH-500",
    "revision": "6e4ed1a2a79af7d8630a6b768ec859cb5af4d3be",
    "file": "test.jsonl",
    "note": "Identical problems/answers to the repo's evaluation/data/math500 (repo adds the 'Return your final response within \\boxed{}. ' prefix, which official_prompt adds too)"
  },
  "selection": {"n_questions": 100, "seed": 20251011, "stratify_by": "level", "per_level": 20},
  "budgets": [500, 1000, 2000, 4000, 6000, 10000],
  "conditions": {
    "A": "No external control tokens (official remaining=default); model may emit control tokens itself",
    "B": "Official BudgetThinker injection, K=8: delta=B//8+1, 7/8..1/8 overwrite the sampled token"
  },
  "prompt_identical_across_conditions": true,
  "sampling": {"temperature": 0.6, "top_p": 0.95, "n_samples": 8,
               "note": "Extended setting requested by the study. The official sampling script passes top_p=0.95 but process_chunk hard-codes top_p=0.9."},
  "greedy_reference": {"temperature": 0.0, "top_p": 1.0, "n_samples": 1, "note": "Paper's MATH-500 setting"},
  "seeds": "seed = sha256(run|q_uid|budget|sample_idx)[:63 bits]; shared by A and B (common random numbers); one CPU torch.Generator per generation",
  "termination": {
    "reasoning_cap": "B + B//50 generated tokens before </think> (official stage-1 max_tokens)",
    "natural_within_budget": "model emits </think> with reasoning_tokens <= B",
    "natural_over_budget": "model emits </think> with B < reasoning_tokens <= B + B//50",
    "forced": "cap reached without </think>: append official suffix, then <= 50 answer tokens",
    "forced_suffix": "\n</think>\n\n**Final Answer**\\boxed",
    "forced_answer_cap": 50,
    "natural_answer_cap": 512,
    "eos_in_think": "model emits EOS before </think> (incomplete)",
    "same_policy_for_A_and_B": true
  },
  "token_counting": {
    "reasoning_tokens": "all generated tokens before </think>: <think>, text, injected and self-generated control tokens",
    "also_logged": ["reasoning_tokens_excl_ctrl", "reasoning_tokens_excl_injected", "ctrl_injected_*", "ctrl_self_*", "suffix_tokens", "answer_tokens", "generated_total"]
  },
  "pilot": {"n_questions": 5, "pick": "first selected question of each level", "budgets": [500, 2000], "n_samples": 2, "conditions": ["A", "B"]},
  "rq1_probe": {"n_questions": 5, "budgets": [500, 2000], "decoding": "greedy", "conditions": ["A", "B"]},
  "rows_per_batch": {"500": 32, "1000": 32, "2000": 32, "4000": 16, "6000": 16, "10000": 16, "default": 16},
  "rows_per_batch_a100": {"500": 256, "1000": 256, "2000": 192, "4000": 128, "6000": 96, "10000": 64, "default": 64},
  "colab": {"gpu": "T4 (pilot, 10-question run); A100 (full 100-question run, approved by the user 2026-10-11)",
            "cu_per_hour_assumed": 1.76, "cu_rate_note": "verify in Colab > Resources; the estimate cell takes the rate as a parameter"}
}

## 3. البيانات واختيار الأسئلة (ثابت وقابل لإعادة الإنتاج)

In [ ]:
import importlib, hashlib
sys.path.insert(0, os.path.join(BASE, "src"))
import bt_engine as E, bt_runner as R, bt_metrics as M
for _m in (E, R, M):
    importlib.reload(_m)
CFG = json.load(open("configs/experiment.json"))
from huggingface_hub import hf_hub_download
DS_PATH = hf_hub_download(CFG["dataset"]["id"], CFG["dataset"]["file"], repo_type="dataset",
                          revision=CFG["dataset"]["revision"])
MATH = R.load_math500(DS_PATH)
assert len(MATH) == 500
QS = R.select_questions(MATH, CFG["selection"]["n_questions"], CFG["selection"]["seed"])
PILOT_QS = [next(q for q in QS if int(q["level"]) == L) for L in range(1, 6)]
SOL = {q["unique_id"]: q["solution"] for q in MATH}
GOLD = {q["unique_id"]: q["answer"] for q in MATH}
json.dump({"dataset": CFG["dataset"], "selection": CFG["selection"],
           "questions": [{"order": i, "unique_id": q["unique_id"], "level": int(q["level"]),
                          "subject": q["subject"], "answer": q["answer"]} for i, q in enumerate(QS)],
           "pilot_unique_ids": [q["unique_id"] for q in PILOT_QS]},
          open("configs/questions_100.json", "w"), indent=1, ensure_ascii=False)
import pandas as pd
print(pd.Series([int(q["level"]) for q in QS]).value_counts().sort_index().to_dict())
pd.DataFrame([{"level": q["level"], "uid": q["unique_id"], "subject": q["subject"], "answer": q["answer"]}
              for q in PILOT_QS])

## 4. تنزيل النموذج والتحقق من هويته وتحميله (bf16 على A100، وfp16 على T4)

In [ ]:
import torch
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer, AutoModelForCausalLM
t0 = time.time()
MODEL_DIR = snapshot_download(CFG["model"]["id"], revision=CFG["model"]["revision"])
T_DOWNLOAD = time.time() - t0
h = hashlib.sha256()
with open(os.path.join(MODEL_DIR, "model.safetensors"), "rb") as f:
    for chunk in iter(lambda: f.read(1 << 24), b""):
        h.update(chunk)
SHA = h.hexdigest()
assert SHA == CFG["model"]["safetensors_sha256"], SHA
tok = AutoTokenizer.from_pretrained(MODEL_DIR)
assert len(tok) == 151672 and tok.bos_token_id == 151646 and tok.eos_token_id == E.EOS
assert tok.convert_tokens_to_ids("<think>") == E.THINK_OPEN and tok.convert_tokens_to_ids("</think>") == E.THINK_CLOSE
for i, t in enumerate(E.CTRL_IDS):
    assert tok.convert_ids_to_tokens(t) == f"\n<remaining>{i + 1}/8</remaining>\n"
assert tok.encode(E.FORCED_SUFFIX_TEXT, add_special_tokens=False) == list(E.FORCED_SUFFIX_IDS)
torch.cuda.reset_peak_memory_stats()
# Published weights are bf16; T4 (sm_75) has no native bf16, so it falls back to fp16.
DTYPE = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(MODEL_DIR, torch_dtype=DTYPE,
                                             attn_implementation="sdpa").cuda().eval()
torch.cuda.synchronize()
T_LOAD = time.time() - t0
VRAM_MODEL = torch.cuda.memory_allocated() / 2**30
META = {"model_id": CFG["model"]["id"], "model_revision": CFG["model"]["revision"],
        "tokenizer_id": CFG["model"]["tokenizer_id"], "tokenizer_revision": CFG["model"]["tokenizer_revision"],
        "dataset_id": CFG["dataset"]["id"], "dataset_revision": CFG["dataset"]["revision"],
        "dtype": str(DTYPE).replace("torch.", ""), "official_commit": OFFICIAL_COMMIT}
import bt_fast as FA
(FA.enable_gqa_decode if FAST else FA.disable_gqa_decode)()
MANIFEST = {**META, "engine": "fast" if FAST else "stock", "safetensors_sha256": SHA, "env": R.env_info(), "download_s": round(T_DOWNLOAD, 1),
            "load_s": round(T_LOAD, 1), "vram_model_gb": round(VRAM_MODEL, 2), "install_s": round(T_INSTALL, 1)}
json.dump(MANIFEST, open("configs/run_manifest.json", "w"), indent=1)
print(json.dumps(MANIFEST, indent=1))

## 5. اختبار أولي (Smoke test)

In [ ]:
if DO_SMOKE:
    q = PILOT_QS[0]
    ids = tok(E.official_prompt(q["problem"], 500)).input_ids
    assert ids[0] == tok.bos_token_id and ids[-1] == 151645
    assert [i for i in ids if i in E.CTRL_SET] == list(range(151671, 151664, -1))  # prompt lists the 7 tokens
    # fp16 vs fp32 on the first 64 greedy tokens (teacher-forced comparison)
    r = E.run_rows(model, [E.RowSpec("fp", ids, 500, False, 0)], E.GenConfig(temperature=0.0), tok, max_steps=64)[0]
    seq = torch.tensor([ids + r["output_ids"]], device="cuda")
    with torch.inference_mode():
        l16 = model(seq).logits[0, len(ids) - 1:-1].float()
    m32 = AutoModelForCausalLM.from_pretrained(MODEL_DIR, torch_dtype=torch.float32, attn_implementation="sdpa").cuda().eval()
    with torch.inference_mode():
        l32 = m32(seq).logits[0, len(ids) - 1:-1].float()
    del m32; torch.cuda.empty_cache()
    p16, p32 = torch.log_softmax(l16, -1), torch.log_softmax(l32, -1)
    FP_CHECK = {"top1_agreement": (l16.argmax(-1) == l32.argmax(-1)).float().mean().item(),
                "max_abs_logit_diff": (l16 - l32).abs().max().item(),
                "mean_kl_fp32_fp16": (p32.exp() * (p32 - p16)).sum(-1).mean().item(),
                "any_nan_fp16": bool(torch.isnan(l16).any())}
    print("fp16 vs fp32:", FP_CHECK)
    print("first 64 tokens:", repr(tok.decode(r["output_ids"])[:300]))
    # one greedy A/B pair at B=500 with full logging
    cfg0 = E.GenConfig(temperature=0.0)
    smoke = E.run_rows(model, [E.RowSpec("smoke|A", ids, 500, False, 0, {**META, "run": "smoke"}),
                               E.RowSpec("smoke|B", ids, 500, True, 0, {**META, "run": "smoke"})], cfg0, tok)
    for s in smoke:
        print(f"\n== condition {s['condition']}: {s['category']} | reasoning {s['counts']['reasoning_tokens']} "
              f"| injected {s['counts']['ctrl_injected_total']} | self {s['counts']['ctrl_self_total']} "
              f"| stop {s['stop_reason']} | {s['timing']['batch_wall_s']}s")
        print("   control events:", [(e['pos'], e['label'], e['source'], round(e['p_model'], 4)) for e in s["ctrl_events"]])
        print("   answer:", repr(s["answer_text"][-160:]))
    json.dump({"fp_check": FP_CHECK, "records": smoke}, open("raw_generations/smoke.json", "w"))

## 6. التجربة الأولى (RQ1): التوليد الذاتي لرموز التحكم — إعدادات الورقة (Greedy, N=1)
5 أسئلة (سؤال من كل مستوى) × ميزانيتان × حالتان = 20 توليداً. تُسجَّل Token IDs الخام مع مصدر كل توكن،
واحتمال النموذج للرمز المتوقع في كل موضع جدولة (حتى في الحالة A حيث لا حقن).

In [ ]:
LOG = R.make_logger(os.path.join(BASE, "logs", "run.log"))
RAW = os.path.join(BASE, "raw_generations")
RPB_KEY = "rows_per_batch_a100" if "A100" in gpu else "rows_per_batch"
RPB = {int(k) if k != "default" else k: v for k, v in CFG[RPB_KEY].items()}
print(RPB_KEY, RPB)
STATS = {}
if DO_RQ1_PROBE:
    cfg_g = E.GenConfig(temperature=0.0, top_p=1.0)
    units = R.make_units(PILOT_QS, CFG["rq1_probe"]["budgets"], ["A", "B"], 1, "rq1_greedy", 0.0, 1.0, tok, META)
    STATS["rq1_greedy"] = R.run_units(model, tok, units, cfg_g, RAW, "rq1_greedy", RPB, LOG, prealloc=FAST)
    rq1 = R.read_units(RAW, "rq1_greedy")
    tab = []
    for r in sorted(rq1, key=lambda x: (x["budget"], x["q_level"], x["condition"])):
        ev = r["ctrl_events"]
        ep = r["expected_ctrl_probs"]
        tab.append({"B": r["budget"], "lvl": r["q_level"], "cond": r["condition"], "category": r["category"],
                    "reason_tok": r["counts"]["reasoning_tokens"], "<think>": r["counts"]["think_open_generated"],
                    "self_ctrl": [(e["pos"], e["label"]) for e in ev if e["source"] == "model"],
                    "n_injected": sum(e["source"] == "injected" for e in ev),
                    "max_p_expected": round(max([x["p"] for x in ep], default=0), 4),
                    "steps_ctrl_mass>1e-3": len(r["ctrl_trace"])})
    display(pd.DataFrame(tab))

## 7. التجربة الأولية Pilot — 40 توليداً (T=0.6, top-p=0.95)

In [ ]:
if DO_PILOT:
    cfg_s = E.GenConfig(temperature=CFG["sampling"]["temperature"], top_p=CFG["sampling"]["top_p"])
    units = R.make_units(PILOT_QS, CFG["pilot"]["budgets"], ["A", "B"], CFG["pilot"]["n_samples"], "pilot",
                         cfg_s.temperature, cfg_s.top_p, tok, META)
    STATS["pilot"] = R.run_units(model, tok, units, cfg_s, RAW, "pilot", RPB, LOG, prealloc=FAST)
    display(pd.DataFrame(STATS["pilot"]["batches"]))

## 8. قياس السرعة والذاكرة (Throughput/Memory probe) — يشمل 16 صفاً بسياق 10,200

In [ ]:
from transformers import DynamicCache
def decode_probe(rows, ctx, steps=24, chunk=512):
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    cache = FA.PreallocCache(ctx + steps + 2) if FAST else DynamicCache()
    with torch.inference_mode():
        for s in range(0, ctx, chunk):
            x = torch.randint(1000, 100000, (rows, min(chunk, ctx - s)), device="cuda")
            model(input_ids=x, past_key_values=cache, use_cache=True, logits_to_keep=1)
        gens = [torch.Generator().manual_seed(i) for i in range(rows)]
        x = torch.randint(1000, 100000, (rows, 1), device="cuda")
        torch.cuda.synchronize(); t0 = time.time()
        for _ in range(steps):
            out = model(input_ids=x, past_key_values=cache, use_cache=True, logits_to_keep=1)
            t, p = E.sample_tokens(out.logits[:, -1].float(), 0.6, 0.95, gens)
            p.index_select(1, torch.tensor(E.CTRL_IDS, device="cuda")).tolist()
            x = t.unsqueeze(1)
        torch.cuda.synchronize()
    dt = (time.time() - t0) / steps
    del cache; torch.cuda.empty_cache()
    return {"rows": rows, "ctx": ctx, "ms_per_step": round(dt * 1000, 1), "tok_per_s": round(rows / dt, 1),
            "peak_gb": round(torch.cuda.max_memory_allocated() / 2**30, 2)}
PROBE = []
if DO_THROUGHPUT_PROBE:
    for rows in ((4, 16, 32, 48) if FAST else (4, 16, 32)):
        for ctx in (512, 2048, 4096, 10200):
            try:
                PROBE.append(decode_probe(rows, ctx))
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache(); PROBE.append({"rows": rows, "ctx": ctx, "ms_per_step": None, "oom": True})
    PROBE = pd.DataFrame(PROBE); PROBE.to_csv(f"metrics/throughput_probe_{'fast' if FAST else 'stock'}.csv", index=False); display(PROBE)

### 8b. التحقق من تكافؤ المسار السريع مع المسار القياسي (Greedy)
تُعاد وحدات RQ1 نفسها (20 توليداً Greedy) بالمسار الآخر، وتُقارن Token IDs توليداً بتوليد.
التطابق التام غير مضمون في fp16 لأن ترتيب العمليات الحسابية يختلف؛ لذلك نسجّل موضع أول اختلاف.

In [ ]:
DO_FAST_VALIDATION = True
if DO_FAST_VALIDATION and DO_RQ1_PROBE:
    other = "stock" if FAST else "fast"
    (FA.disable_gqa_decode if FAST else FA.enable_gqa_decode)()
    units_v = R.make_units(PILOT_QS, CFG["rq1_probe"]["budgets"], ["A", "B"], 1, "rq1_greedy", 0.0, 1.0, tok, META)
    STATS["rq1_greedy_" + other] = R.run_units(model, tok, units_v, E.GenConfig(temperature=0.0, top_p=1.0), RAW,
                                               "rq1_greedy_" + other, RPB, LOG, prealloc=not FAST)
    (FA.enable_gqa_decode if FAST else FA.disable_gqa_decode)()
    a = {r["gen_id"].split("|", 1)[1]: r for r in R.read_units(RAW, "rq1_greedy")}
    b = {r["gen_id"].split("|", 1)[1]: r for r in R.read_units(RAW, "rq1_greedy_" + other)}
    rows_v = []
    for k in sorted(a):
        x, y = a[k]["output_ids"], b[k]["output_ids"]
        n = next((i for i, (u, v) in enumerate(zip(x, y)) if u != v), min(len(x), len(y)))
        rows_v.append({"gen": k, "identical": x == y, "first_diff": None if x == y else n, "len_a": len(x), "len_b": len(y),
                       "same_category": a[k]["category"] == b[k]["category"]})
    VAL = pd.DataFrame(rows_v); VAL.to_csv("metrics/fast_path_validation.csv", index=False)
    print(f"identical {VAL.identical.sum()}/{len(VAL)} | same category {VAL.same_category.sum()}/{len(VAL)} | "
          f"median first divergence (non-identical): {VAL.first_diff.dropna().median()}")
    display(VAL)

## 9. التصحيح والمقاييس للتجارب الأولية

In [ ]:
def analyze(run_names, tag, reps=2000):
    recs = [r for rn in run_names for r in R.read_units(RAW, rn)]
    off = M.grade_official(recs, tok, SOL, f"{OFFICIAL}/evaluation", GRADER_ENV, os.path.join(BASE, "metrics", "grading", tag))
    mv = M.grade_math_verify(recs, tok, GOLD)
    df = M.flatten(recs, off, mv)
    summ, pab, ctrl = M.summarize(df, reps), M.paired_ab(df, reps), M.ctrl_positions(recs)
    df.to_csv(f"metrics/{tag}_generations.csv", index=False)
    summ.to_csv(f"metrics/{tag}_summary.csv", index=False)
    pab.to_csv(f"metrics/{tag}_paired_AB.csv", index=False)
    ctrl.to_csv(f"metrics/{tag}_ctrl_tokens.csv", index=False)
    M.expected_prob_table(recs).to_csv(f"metrics/{tag}_expected_ctrl_probs.csv", index=False)
    return recs, df, summ, pab, ctrl
pre = [rn for rn, flag in (("rq1_greedy", DO_RQ1_PROBE), ("pilot", DO_PILOT)) if flag]
RECS, DF, SUMM, PAB, CTRL = analyze(pre, "preliminary")
cols = ["run", "budget", "condition", "n_gen", "BFR", "BFR_slack", "FTR", "EOS_in_think", "BUR", "BUR_total",
        "MeanReasoning", "Accuracy", "Answer_complete", "mean_ctrl_self", "frac_gen_with_self_ctrl",
        "mean_ctrl_injected", "suppressed_think_close_rate", "pred_not_boxed", "grader_disagree"]
display(SUMM[cols].round(3))
display(DF.groupby(["run", "budget", "condition", "category"]).size().unstack(fill_value=0))
if DO_PILOT:
    for p in M.make_plots(SUMM, DF, CTRL, "reports/figures", "pilot"):
        from IPython.display import Image; display(Image(p, width=520))

## 10. تقدير زمن وتكلفة التشغيل الكامل

In [ ]:
import numpy as np
PROBE_OK = PROBE.dropna(subset=["ms_per_step"])
def ms_step(rows, ctx):
    """Per-step decode time from the probe grid (next larger row count -> conservative)."""
    cands = sorted(PROBE_OK["rows"].unique())
    r = next((x for x in cands if x >= rows), cands[-1])
    sub = PROBE_OK[PROBE_OK["rows"] == r].sort_values("ctx")
    return float(np.interp(ctx, sub["ctx"], sub["ms_per_step"])) * (rows / r if rows > r else 1.0)
RNG = np.random.default_rng(0)
def batch_hours(b, rows, ratios, sims=20, dt=25):
    """Monte Carlo batch time: rows finish at sampled lengths and are pruned (batch shrinks)."""
    cap, hrs = b + b // 50, []
    for _ in range(sims):
        L = np.minimum(RNG.choice(ratios, size=rows) * b, cap) + 60   # + suffix/answer tokens
        steps = np.arange(0, int(L.max()), dt)
        hrs.append(sum(ms_step(max(1, int((L > t).sum())), 150 + t) for t in steps) * dt / 1000 / 3600)
    return float(np.mean(hrs))
# Length scenarios: "expected" uses the pilot's empirical reasoning/B ratios (both conditions);
# budgets without pilot data reuse the B=2000 ratios (conservative: ratios usually fall as B grows).
# "upper" assumes every generation runs to the cap.
pil = DF[DF["run"] == "pilot"] if DO_PILOT else DF
emp = {b: pil[pil["budget"] == b]["reasoning_ratio"].values for b in pil["budget"].unique()}
proxy = emp.get(2000, np.array([1.0]))
rows_est = []
for run, n_samp in (("greedy_ref", 1), ("sampled", 8)):
    for b in CFG["budgets"]:
        rows_per_unit = 2 * n_samp
        rpb = RPB.get(b, 16)
        units_per_batch = max(1, rpb // rows_per_unit)
        rows_b = rows_per_unit * units_per_batch
        n_batches = int(np.ceil(100 / units_per_batch))
        e = batch_hours(b, rows_b, emp.get(b, proxy))
        u = batch_hours(b, rows_b, np.array([1.0]), sims=1)
        rows_est.append({"run": run, "budget": b, "batches": n_batches, "rows/batch": rows_b,
                         "ratio_source": "pilot" if b in emp else "B=2000 proxy",
                         "hours_expected": n_batches * e, "hours_upper": n_batches * u})
EST = pd.DataFrame(rows_est)
tot = EST.groupby("run")[["hours_expected", "hours_upper"]].sum()
tot["CU_expected"] = tot["hours_expected"] * CU_PER_HOUR
tot["CU_upper"] = tot["hours_upper"] * CU_PER_HOUR
EST.to_csv("metrics/cost_estimate_by_budget.csv", index=False); tot.to_csv("metrics/cost_estimate_total.csv")
display(EST.round(2)); display(tot.round(2))
raw_mb = sum(os.path.getsize(os.path.join(dp, f)) for dp, _, fs in os.walk(RAW) for f in fs) / 2**20
n_prelim = len(RECS)
print(f"raw data so far: {raw_mb:.1f} MB for {n_prelim} generations -> ~{raw_mb / max(n_prelim, 1) * 10800 / 1024:.2f} GB for 10,800")
print(f"session elapsed: {(time.time() - T_SESSION0) / 60:.1f} min (~{(time.time() - T_SESSION0) / 3600 * CU_PER_HOUR:.2f} CU at {CU_PER_HOUR}/h)")

---
## ⛔ نقطة توقف: لا يبدأ التشغيل الكامل إلا بموافقة صريحة
راجِع تقرير الـPilot وتقدير التكلفة أعلاه. التشغيل الكامل = مرجع Greedy (1,200 توليد) + التجربة الموسّعة (9,600 توليد).
لتشغيله: اجعل `APPROVED_FULL_RUN = True` في خلية الإعدادات ثم شغّل الخلايا التالية. يُستأنف تلقائياً بعد أي انقطاع.

In [ ]:
if not APPROVED_FULL_RUN:
    raise SystemExit("Full run not approved (APPROVED_FULL_RUN=False). Stopping here.")
FULL = {"greedy_ref": (1, 0.0, 1.0), "sampled": (CFG["sampling"]["n_samples"], CFG["sampling"]["temperature"], CFG["sampling"]["top_p"])}
for run in FULL_RUNS:
    n_samp, temp, top_p = FULL[run]
    cfg_r = E.GenConfig(temperature=temp, top_p=top_p)
    for chunk in range(0, len(QS), 20):  # 20 questions at a time -> balanced partial results
        units = R.make_units(QS[chunk:chunk + 20], CFG["budgets"], ["A", "B"], n_samp, run, temp, top_p, tok, META)
        STATS.setdefault(run, []).append(R.run_units(model, tok, units, cfg_r, RAW, run, RPB, LOG, prealloc=FAST))

In [ ]:
RECS_F, DF_F, SUMM_F, PAB_F, CTRL_F = analyze([r for r in FULL_RUNS], "full")
display(SUMM_F[cols].round(3))
display(PAB_F.round(4))
for run in FULL_RUNS:
    for p in M.make_plots(SUMM_F, DF_F, CTRL_F, "reports/figures", run):
        from IPython.display import Image; display(Image(p, width=520))

## 11. تنزيل البيانات إلى الحاسوب
يجمع كل المخرجات في ملف zip واحد مع `MANIFEST.json` (SHA-256 لكل ملف)، ثم ينزّله عبر المتصفح:
البيانات الخام (`raw_generations/*.jsonl.gz`)، والجداول (`metrics/`)، والرسوم (`reports/figures/`)،
والإعدادات (`configs/`)، والسجل (`logs/`)، والكود (`src/`)، ونسخة من هذا الـNotebook.
يمكن تشغيله أثناء التجربة: الوحدات المكتملة فقط تُضمَّن. اجعل `DOWNLOAD = False` لإنشاء الملف دون تنزيله.

In [ ]:
%%writefile src/bt_export.py
"""Package every study output into one zip (with a SHA-256 manifest) and download it locally.

Run in Colab after (or during) a run:  %run -i src/bt_export.py
Only completed units are included: unit files are written atomically, *.tmp files are skipped.
Set DOWNLOAD = False beforehand to build the zip without triggering the browser download.
"""
import hashlib
import json
import os
import time
import zipfile

BASE = globals().get("BASE", "/content/BudgetThinker_eval")
DOWNLOAD = globals().get("DOWNLOAD", True)
INCLUDE = ["configs", "raw_generations", "metrics", "reports", "logs", "src"]
NOTEBOOK = "/content/bt_bundle/notebooks/BudgetThinker_Independent_Eval.ipynb"


def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


stamp = time.strftime("%Y%m%d_%H%M%SZ", time.gmtime())
out = f"/content/BudgetThinker_eval_results_{stamp}.zip"
files_meta, n_units = [], 0
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for d in INCLUDE:
        for dp, _, fs in os.walk(os.path.join(BASE, d)):
            for f in sorted(fs):
                if f.endswith(".tmp"):
                    continue
                p = os.path.join(dp, f)
                rel = os.path.relpath(p, BASE).replace(os.sep, "/")
                z.write(p, rel)
                files_meta.append({"path": rel, "bytes": os.path.getsize(p), "sha256": _sha256(p)})
                n_units += f.endswith(".jsonl.gz")
    if os.path.exists(NOTEBOOK):
        z.write(NOTEBOOK, "notebooks/BudgetThinker_Independent_Eval.ipynb")
    z.writestr("MANIFEST.json", json.dumps({
        "created_utc": stamp, "base": BASE, "n_files": len(files_meta),
        "n_generation_units": n_units, "files": files_meta}, indent=1))

with zipfile.ZipFile(out) as z:
    assert z.testzip() is None, "zip integrity check failed"
size_mb = os.path.getsize(out) / 2**20
print(f"{out}\n{len(files_meta)} files, {n_units} generation units, {size_mb:.1f} MB, sha256 {_sha256(out)[:16]}")

if DOWNLOAD:
    try:
        from google.colab import files
        files.download(out)
    except ImportError:
        print("Not running in Colab: zip kept at", out)

In [ ]:
%run -i src/bt_export.py